# Content Creation at Scale

## Initial Imports

In [51]:
# warning control
import warnings
warnings.filterwarnings("ignore")

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

import litellm
litellm.set_verbose = True
os.environ['LITELLM_LOG'] = 'DEBUG'

## Creating Structred Output

In [52]:
from pydantic import BaseModel, Field
from typing import List

class SocialMediaPost(BaseModel):
    """
    Model for social media post
    """
    platform: str = Field(..., description="The social media platform where the post will be published (e.g., Twitter, Linkedin).")
    content: str = Field(..., description="The content of the social media post, including any hashtags or mentions.")

class ContentOutput(BaseModel):
    """
    Model for content output
    """
    article: str = Field(..., description="The article formatted in markdown.")
    social_media_posts: List[SocialMediaPost] = Field(..., description="A list of social media posts related to the article.")


## Loading Tasks and Agents YAML files

In [53]:
# Define file paths for YAML configurations
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml',
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

## Importing CrewAI Tools

In [54]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool, WebsiteSearchTool

In [55]:
from crewai import LLM


mistral_model = 'mistral/mistral-large-latest'
# groq_llm = "groq/llama-3.3-70b-versatile"
groq_llm = "groq/mistral-saba-24b"

tools_llm_configs = {
    "small": {
        "provider":"mistralai",
        "config":{
            "model":"mistral/mistral-small-latest",
            'api_key': os.getenv("MISTRAL_API_KEY"),
        },
    },
    "large": {
        "provider":"mistralai",
        "config":{
            "model":"mistral/mistral-large-latest",
            'api_key': os.getenv("MISTRAL_API_KEY"),
        }},
}

llms = {
    "small": LLM(
        model="mistral/mistral-small-latest",
        provider="mistralai",
        config={
            "api_key": os.getenv("MISTRAL_API_KEY")
        }
    ),
    "large": LLM(
        model="mistral/mistral-large-latest",
        provider="mistralai",
        config={
            "api_key": os.getenv("MISTRAL_API_KEY")
        }
    ),
}

## Creating Crews, Agents and Tasks

### Creating Agents

In [56]:
def create_website_search_tool(llm_config):
    return WebsiteSearchTool(config={
        "llm": llm_config,
        "embedder": {
            "provider": "mistralai",
            "config": {
                "model": "mistral-embed",
                "api_key": os.getenv("MISTRAL_API_KEY")
            }
        }
    })

In [77]:
market_news_monitor_agent = Agent(
    config=agents_config['market_news_monitor_agent'],
    tools=[SerperDevTool(), ScrapeWebsiteTool()],
    llm=mistral_model,
)

data_analyst_agent = Agent(
    config=agents_config['data_analyst_agent'],
    tools=[
        SerperDevTool(), 
        WebsiteSearchTool(
            config={
                "llm": {
                    "config":{
                    "model": "mistral/mistral-small-latest",
                    "api_key": os.getenv("MISTRAL_API_KEY")
                    },
                },
                "embedder": {
                "config": {
                    "model": "mistral-embed",
                    "api_key": os.getenv("MISTRAL_API_KEY")
                    }
                }
            }
        )],
llm=mistral_model,
)

content_creator_agent = Agent(
    config=agents_config['content_creator_agent'],
    tools=[
        SerperDevTool(), 
        WebsiteSearchTool(
            config={
                "llm": {
                    "config":{
                    "model": "mistral/mistral-small-latest",
                    "api_key": os.getenv("MISTRAL_API_KEY")
                    },
                },
                "embedder": {
                "config": {
                    "model": "mistral-embed",
                    "api_key": os.getenv("MISTRAL_API_KEY")
                    }
                }
            }
        )
        ],
    llm=mistral_model
)

quality_assurance_agent = Agent(
    config=agents_config['quality_assurance_agent'],
    llm=mistral_model
)

### Creating Tasks

In [78]:

monitor_financial_news_task = Task(
    config=tasks_config['monitor_financial_news'],
    agent=market_news_monitor_agent
)

analyze_market_data_task = Task(
    config=tasks_config['analyze_market_data'],
    agent=data_analyst_agent
)

create_content_task = Task(
    config=tasks_config['create_content'],
    agent=content_creator_agent,
    context=[monitor_financial_news_task, analyze_market_data_task]
)

quality_assurance_task = Task(
    config=tasks_config['quality_assurance'],
    agent=quality_assurance_agent,
    output_pydantic=ContentOutput
)

### Creating Crew

In [79]:
content_creation_crew = Crew(
    agents=[
        market_news_monitor_agent,
        data_analyst_agent,
        content_creator_agent,
        quality_assurance_agent
        ],
    tasks=[
        monitor_financial_news_task,
        analyze_market_data_task,
        create_content_task,
        quality_assurance_task
    ],
    verbose=True
)

In [76]:
from litellm import completion
import os

os.environ["LITELLM_DEBUG"] = "True"  # Enable debug logging

models = {
    "small": "mistral/mistral-small-latest",
    "large": "mistral/mistral-large-latest"
}

for name, model in models.items():
    print(f"\n🔍 Testing {name} model: {model}")
    try:
        response = completion(
            model=model,
            api_key=os.getenv("MISTRAL_API_KEY"),
            messages=[{"role": "user", "content": "Say hello in 3 words"}]
        )
        content = response["choices"][0]["message"]["content"]
        print(f"✅ {name.capitalize()} Model Response: {content}")
    except Exception as e:
        print(f"❌ {name.capitalize()} Model Error:", str(e))

11:41:16 - LiteLLM:WARNING: utils.py:428 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.
2025-04-10 11:41:16,561 - 40852 - utils.py-utils:428 - WARNING: `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.



🔍 Testing small model: mistral/mistral-small-latest
Initialized litellm callbacks, Async Success Callbacks: [<crewai.utilities.token_counter_callback.TokenCalcHandler object at 0x0000022369DE1150>]
SYNC kwargs[caching]: False; litellm.cache: None; kwargs.get('cache')['no-cache']: False
Final returned optional params: {'extra_body': {}}


11:41:17 - LiteLLM:WARNING: utils.py:428 - `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.
2025-04-10 11:41:17,154 - 40852 - utils.py-utils:428 - WARNING: `litellm.set_verbose` is deprecated. Please set `os.environ['LITELLM_LOG'] = 'DEBUG'` for debug logs.


RAW RESPONSE:
{"id": "dbc4d754223342ae9d2c69a7f8b0909c", "choices": [{"finish_reason": "stop", "index": 0, "logprobs": null, "message": {"content": "Hello! How are you?", "refusal": null, "role": "assistant", "annotations": null, "audio": null, "function_call": null, "tool_calls": null}}], "created": 1744265477, "model": "mistral-small-latest", "object": "chat.completion", "service_tier": null, "system_fingerprint": null, "usage": {"completion_tokens": 6, "prompt_tokens": 9, "total_tokens": 15, "completion_tokens_details": null, "prompt_tokens_details": null}}


✅ Small Model Response: Hello! How are you?

🔍 Testing large model: mistral/mistral-large-latest
Initialized litellm callbacks, Async Success Callbacks: [<crewai.utilities.token_counter_callback.TokenCalcHandler object at 0x0000022369DE1150>]
SYNC kwargs[caching]: False; litellm.cache: None; kwargs.get('cache')['no-cache']: False
Final returned optional params: {'extra_body': {}}
RAW RESPONSE:
{"id": "f41d199a85584cefbb825af1b2

## Kicking off the Crew

In [80]:
result = content_creation_crew.kickoff(inputs={
    'subject': 'US tariff on China',
})

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c57f49a8-c827-4eaf-ac9c-7fc7873997df                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 42163a8e-64e1-4837-8890-35e84b02ede9
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 42163a8e-64e1-4837-8890-35e84b02ede9
       Status: Executing Task...
    └── 🤖 Agent: Lead Market Analyst
        
            Status: In Progress

# Agent: Lead Market Analyst
## Task: Monitor and analyze the latest news and updates related to the financial markets, with a particular focus on US tariff on China. Identify and summarize the most relevant and impactful news items that could influence market trends or investor decisions. Utilize financial news APIs and real-time market data tools to gather up-to-date information. Focus on detecting trends, regulatory changes, or significant economic indicators that directly relate to US tariff on China.



🚀 Crew: crew
└── 📋 Task: 42163a8e-64e1-4837-8890-35e84b02ede9
       Status: Executing Task...
    └── 🤖 Agent: Lead Market Analyst
        
            Status: In Progress
        └── 🧠 Thinking...

...



# Agent: Chief Content Officer
## Final Answer: 
{
  "article": "# Comprehensive Analysis Report: Impact of US Tariffs on China and Market Reactions

## Executive Summary:
The implementation of US tariffs on China has led to significant market volatility and geopolitical tensions. China's retaliatory measures, along with global responses, highlight the far-reaching impacts of these trade actions. This report analyzes key market trends, investor sentiment, and economic indicators to provide actionable insights and recommendations for content creation.

## Key Findings:

### 1. Market Volatility and Sentiment:
The announcement of a 90-day pause on tariffs for most countries, excluding China, has resulted in substantial market volatility. China's retaliatory tariffs, raising levies on US goods to 84%, have exacerbated market instability. Investor sentiment has been negatively impacted, with concerns over disrupted supply chains, increased inflation, and potential economic slowdown.

Mar

🚀 Crew: crew
├── 📋 Task: 42163a8e-64e1-4837-8890-35e84b02ede9
│      Assigned to: Lead Market Analyst
│   
│      Status: ✅ Completed
│   └── 🤖 Agent: Lead Market Analyst
│       
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (1)
│       └── 🔧 Used Read website content (1)
├── 📋 Task: a5034006-9aa6-4290-bed5-5077cfb8536d
│      Assigned to: Chief Data Strategist
│   
│      Status: ✅ Completed
│   └── 🤖 Agent: Chief Data Strategist
│       
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (4)
│       └── 🔧 Failed Search in a specific website (2)
├── 📋 Task: 7c7e35cf-cb36-43a4-aff6-95e15c722f3b
│      Assigned to: Creative Content Director
│   
│      Status: ✅ Completed
│   └── 🤖 Agent: Creative Content Director
│       
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (5)
│       └── 🔧 Failed Search in a specific website (3)
└── 📋 Task: bf039b7f-b1df-461a-a897-ef532ad7b4ea
       Status: Executing Task...
    └── 🤖 Agent: Chief Content Officer
        
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 42163a8e-64e1-4837-8890-35e84b02ede9
│      Assigned to: Lead Market Analyst
│   
│      Status: ✅ Completed
│   └── 🤖 Agent: Lead Market Analyst
│       
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (1)
│       └── 🔧 Used Read website content (1)
├── 📋 Task: a5034006-9aa6-4290-bed5-5077cfb8536d
│      Assigned to: Chief Data Strategist
│   
│      Status: ✅ Completed
│   └── 🤖 Agent: Chief Data Strategist
│       
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (4)
│       └── 🔧 Failed Search in a specific website (2)
├── 📋 Task: 7c7e35cf-cb36-43a4-aff6-95e15c722f3b
│      Assigned to: Creative Content Director
│   
│      Status: ✅ Completed
│   └── 🤖 Agent: Creative Content Director
│       
│           Status: ✅ Completed
│       ├── 🔧 Used Search the internet with Serper (5)
│       └── 🔧 Failed Search in a specific website (3)
└── 📋 Task: bf039b7f-b1df-461a-a897-ef532ad7b4ea
       Assigned to: Chief Content Officer
    
       Status: ✅ Completed
    └── 🤖 Agent: Chief Content Officer
        
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: bf039b7f-b1df-461a-a897-ef532ad7b4ea                                                                     │
│  Agent: Chief Content Officer                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c57f49a8-c827-4eaf-ac9c-7fc7873997df                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Social Content

In [81]:
import textwrap

posts = result.pydantic.dict()['social_media_posts']
for post in posts:
    platform = post['platform']
    content = post['content']
    print(platform)
    wrapped_content = textwrap.fill(content, width=50)
    print(wrapped_content)
    print('-' * 50)

Twitter
The US-China tariff war continues to escalate,
with China retaliating with 84% tariffs on US
goods. Stay informed with our latest analysis on
market volatility and economic impact. #USTariffs
#ChinaTariffs #TradeWar
--------------------------------------------------
Twitter
Investor sentiment takes a hit as US tariffs on
China fuel market volatility. Learn how different
sectors are adapting and what it means for your
investments. #InvestorInsights #MarketVolatility
--------------------------------------------------
Twitter
The global response to US-China tariffs highlights
the need for strategic content creation. Stay
tuned for our in-depth analysis and actionable
insights. #GlobalTrade #EconomicImpact
--------------------------------------------------
LinkedIn
The implementation of US tariffs on China has led
to significant market volatility and geopolitical
tensions. Our latest blog post provides a
comprehensive analysis of the economic impact and
sector-specific challenges. 

## Blog Post

In [82]:
from IPython.display import display, Markdown
display(Markdown(result.pydantic.dict()['article']))

# Comprehensive Analysis Report: Impact of US Tariffs on China and Market Reactions

## Executive Summary:
The implementation of US tariffs on China has led to significant market volatility and geopolitical tensions. China's retaliatory measures, along with global responses, highlight the far-reaching impacts of these trade actions. This report analyzes key market trends, investor sentiment, and economic indicators to provide actionable insights and recommendations for content creation.

## Key Findings:

### 1. Market Volatility and Sentiment:
The announcement of a 90-day pause on tariffs for most countries, excluding China, has resulted in substantial market volatility. China's retaliatory tariffs, raising levies on US goods to 84%, have exacerbated market instability. Investor sentiment has been negatively impacted, with concerns over disrupted supply chains, increased inflation, and potential economic slowdown.

Market volatility has been particularly pronounced in sectors directly affected by the tariffs. The automotive industry, for instance, has seen significant price increases due to higher costs of imported materials. Similarly, the steel and aluminum sectors have faced challenges as tariffs have driven up the prices of raw materials, making domestic production less competitive. These disruptions have led to a ripple effect across global supply chains, affecting manufacturers and consumers alike.

### 2. Economic Impact:
The US tariffs are predicted to cause a 2.4% decline in China's economic growth, affecting exports and domestic consumer sentiment. The tit-for-tat tariff escalation threatens to reduce trade between the US and China by 80%, with broader economic consequences expected. The long-term implications include potential job losses, reduced investment, and a slowdown in economic growth for both countries.

The economic impact is not limited to China and the US. Other countries, particularly those involved in global supply chains, are also feeling the effects. For example, countries that supply components to Chinese manufacturers are experiencing reduced demand, which in turn affects their economic growth. The interconnected nature of global trade means that the repercussions of the US-China tariff dispute are felt worldwide.

### 3. Sector-Specific Impacts:
The automotive, steel, and aluminum sectors are particularly affected, with US farmers and small businesses facing significant challenges. The tariffs have forced businesses to increase prices, making products less competitive and impacting consumer surplus. Small businesses, which often lack the resources to absorb higher costs, are particularly vulnerable to these changes.

US farmers, who rely heavily on exports to China, have been hit hard by the retaliatory tariffs. The agricultural sector has seen a significant drop in demand for products like soybeans, pork, and other commodities. This has led to financial strain for farmers and has highlighted the need for diversification in export markets. Small businesses, which often rely on imported materials, have also struggled with increased costs and reduced profit margins.

### 4. Global Reactions:
Countries are engaging in negotiations to mitigate the effects of the tariffs, while the EU and other nations implement retaliatory measures. The situation underscores the need for timely and insightful content to help businesses and investors navigate the evolving trade landscape. The global response to the US-China tariff dispute highlights the complex and far-reaching impacts of these trade actions.

The EU, for instance, has implemented its own set of retaliatory tariffs on US goods, targeting products like bourbon, motorcycles, and orange juice. Other countries, such as Canada and Mexico, have also taken steps to protect their domestic industries from the effects of the tariffs. These measures underscore the interconnected nature of global trade and the potential for wider economic disruptions.

## Data Visualizations:
[Insert Infographic: Timeline of US-China Tariff Escalation]

[Insert Chart: Impact of Tariffs on Key Economic Indicators]

[Insert Graph: Sector-Specific Price Increases Due to Tariffs]

## Recommendations for Content Creation:

### 1. In-Depth Market Analysis:
Develop content that delves into the sector-specific impacts of the tariffs, highlighting how different industries are adapting to the changes. Create visualizations that illustrate the correlation between tariff implementations and market movements to provide clear insights for investors. For example, a detailed analysis of the automotive sector could include case studies of companies that have successfully navigated the tariffs by diversifying their supply chains or adopting new technologies.

### 2. Investor Sentiment and Strategy:
Produce articles and reports that analyze investor sentiment and provide strategies for navigating market volatility. Include expert opinions and case studies of companies that have successfully managed similar trade disruptions. For instance, a report could feature interviews with financial analysts who offer insights into how investors can protect their portfolios during times of market uncertainty.

### 3. Economic Indicators and Forecasts:
Publish regular updates on key economic indicators affected by the tariffs, such as GDP growth, inflation rates, and trade volumes. Offer forecasts and scenarios that explore the potential long-term implications of the trade war on global economic stability. For example, a quarterly update could provide data on how the tariffs are affecting economic growth in both the US and China, along with projections for the coming year.

### 4. Global Trade Dynamics:
Create content that examines the global reactions to the US-China tariff dispute, including the retaliatory measures taken by other countries. Analyze the broader geopolitical implications and the potential for multilateral trade agreements to mitigate the effects of the tariffs. For instance, an article could explore how the EU's retaliatory tariffs are affecting US exports and the potential for trade negotiations to resolve the dispute.

## Conclusion:
The US tariffs on China have set off a chain reaction of economic and market disruptions that require careful analysis and strategic content creation. By synthesizing complex market data into actionable insights, financial advisory firms can provide valuable guidance to businesses and investors navigating this uncertain landscape. Continuous monitoring and timely updates will be crucial in helping stakeholders make informed decisions and adapt to the evolving trade environment.

## References:
- [China and the Impact of “Liberation Day” Tariffs - CSIS](link_to_csis)
- [The impact of US tariffs on China: three scenarios](link_to_scenarios)
- [China's Comprehensive Retaliation Against U.S. Tariffs | Insights](link_to_insights)
- [China's new tariffs on US goods take effect as Asian markets rally ...](link_to_asian_markets)
- [Trump Tariffs Live Updates: China Hit With 104% Tariffs as Stocks ...](link_to_live_updates)
- [China supply chain diversification challenged by Trump's tariffs](link_to_supply_chain)
- [Trump tariffs live updates: China hits back by raising tariff on US ...](link_to_tariff_updates)
- [Trump Tariffs: The Economic Impact of the Trump Trade War](link_to_economic_impact)
- [Wall Street rebounds sharply after Trump announces 90-day tariff ...](link_to_wall_street)
- [China–United States trade war - Wikipedia](link_to_wikipedia)

---

## Blog Post: Navigating the US-China Tariff Landscape: Insights and Market Reactions

### Introduction:
The recent implementation of US tariffs on China has sent shockwaves through global markets, leading to significant volatility and geopolitical tensions. As China retaliates with its own set of tariffs, the trade war between the two economic giants continues to escalate. This blog post delves into the key findings and market reactions, providing investors and businesses with actionable insights to navigate the evolving trade landscape.

### Market Volatility and Sentiment:
The announcement of a 90-day pause on tariffs for most countries, excluding China, has resulted in substantial market volatility. China's retaliatory tariffs, raising levies on US goods to 84%, have exacerbated market instability. Investor sentiment has been negatively impacted, with concerns over disrupted supply chains, increased inflation, and potential economic slowdown.

Market volatility has been particularly pronounced in sectors directly affected by the tariffs. The automotive industry, for instance, has seen significant price increases due to higher costs of imported materials. Similarly, the steel and aluminum sectors have faced challenges as tariffs have driven up the prices of raw materials, making domestic production less competitive. These disruptions have led to a ripple effect across global supply chains, affecting manufacturers and consumers alike.

### Economic Impact:
The US tariffs are predicted to cause a 2.4% decline in China's economic growth, affecting exports and domestic consumer sentiment. The tit-for-tat tariff escalation threatens to reduce trade between the US and China by 80%, with broader economic consequences expected. The automotive, steel, and aluminum sectors are particularly affected, with US farmers and small businesses facing significant challenges.

The economic impact is not limited to China and the US. Other countries, particularly those involved in global supply chains, are also feeling the effects. For example, countries that supply components to Chinese manufacturers are experiencing reduced demand, which in turn affects their economic growth. The interconnected nature of global trade means that the repercussions of the US-China tariff dispute are felt worldwide.

### Sector-Specific Impacts:
The tariffs have forced businesses to increase prices, making products less competitive and impacting consumer surplus. Countries are engaging in negotiations to mitigate the effects of the tariffs, while the EU and other nations implement retaliatory measures. The situation underscores the need for timely and insightful content to help businesses and investors navigate the evolving trade landscape.

US farmers, who rely heavily on exports to China, have been hit hard by the retaliatory tariffs. The agricultural sector has seen a significant drop in demand for products like soybeans, pork, and other commodities. This has led to financial strain for farmers and has highlighted the need for diversification in export markets. Small businesses, which often rely on imported materials, have also struggled with increased costs and reduced profit margins.

### Global Reactions:
The global response to the US-China tariff dispute highlights the complex and far-reaching impacts of these trade actions. The situation underscores the need for timely and insightful content that helps businesses and investors navigate the evolving trade landscape and understand the long-term implications for global economic stability.

The EU, for instance, has implemented its own set of retaliatory tariffs on US goods, targeting products like bourbon, motorcycles, and orange juice. Other countries, such as Canada and Mexico, have also taken steps to protect their domestic industries from the effects of the tariffs. These measures underscore the interconnected nature of global trade and the potential for wider economic disruptions.

### Data Visualizations:
[Insert Infographic: Timeline of US-China Tariff Escalation]

[Insert Chart: Impact of Tariffs on Key Economic Indicators]

[Insert Graph: Sector-Specific Price Increases Due to Tariffs]

### Conclusion:
The US tariffs on China have set off a chain reaction of economic and market disruptions that require careful analysis and strategic content creation. By synthesizing complex market data into actionable insights, financial advisory firms can provide valuable guidance to businesses and investors navigating this uncertain landscape. Continuous monitoring and timely updates will be crucial in helping stakeholders make informed decisions and adapt to the evolving trade environment.



In [83]:
import os

article_content = result.pydantic.dict()['article']
file_name = "generated_blog.md"

with open(file_name, "w", encoding="utf-8") as f:
    f.write(article_content)

print(f"✅ Blog saved as {file_name}")

✅ Blog saved as generated_blog.md


In [87]:
blog_post = result['article'].split('## Blog Post: ')[1]
print(blog_post)

Navigating the US-China Tariff Landscape: Insights and Market Reactions

### Introduction:
The recent implementation of US tariffs on China has sent shockwaves through global markets, leading to significant volatility and geopolitical tensions. As China retaliates with its own set of tariffs, the trade war between the two economic giants continues to escalate. This blog post delves into the key findings and market reactions, providing investors and businesses with actionable insights to navigate the evolving trade landscape.

### Market Volatility and Sentiment:
The announcement of a 90-day pause on tariffs for most countries, excluding China, has resulted in substantial market volatility. China's retaliatory tariffs, raising levies on US goods to 84%, have exacerbated market instability. Investor sentiment has been negatively impacted, with concerns over disrupted supply chains, increased inflation, and potential economic slowdown.

Market volatility has been particularly pronounced i